<a href="https://colab.research.google.com/github/ekonjmrivas-devops/llm_engineering/blob/mis-ejercicios/week7/Semana_7_d%C3%ADa_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Predecir precios de productos

### Semana 7 Día 1

Introducción a LoRA y QLoRA

In [1]:
# pip installs
# peft - ajuste fino de parámetros

!pip install -q datasets==2.21.0 requests torch peft bitsandbytes transformers==4.43.1 trl accelerate sentencepiece

In [1]:
# imports

import os
# Proporciona funciones para interactuar con el sistema operativo (ej. acceder a variables de entorno)

import re
# Módulo de expresiones regulares para búsqueda y manipulación de patrones en texto

import math
# Funciones matemáticas básicas (sin, cos, sqrt, etc.)

from tqdm import tqdm
# Barra de progreso visual para loops — muestra avance y tiempo estimado durante iteraciones

from google.colab import userdata
# Acceso a secretos guardados en Google Colab (ej. tokens de autenticación) de forma segura

from huggingface_hub import login
# Autenticarse en Hugging Face Hub usando token de API — permite descargar modelos privados y subir resultados

import torch
# Framework de deep learning de PyTorch — gestión de tensores, GPU computing, autograd para gradientes

import transformers
# Librería de Hugging Face con modelos pre-entrenados (BERT, GPT, Llama, etc.) y utilidades

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, set_seed
# Funciones específicas de transformers:
#   - AutoModelForCausalLM: carga automáticamente modelos generativos (ej. Llama 3.1) según el nombre
#   - AutoTokenizer: carga automáticamente el tokenizador correspondiente al modelo
#   - BitsAndBytesConfig: configuración para cuantización de 8 bits y 4 bits (NF4) durante carga del modelo
#   - TrainingArguments: especifica parámetros de entrenamiento (learning rate, batch size, épocas, etc.)
#   - set_seed: fija la semilla aleatoria para reproducibilidad

from peft import LoraConfig, PeftModel
# PEFT = Parameter-Efficient Fine-Tuning (Hugging Face):
#   - LoraConfig: define la configuración de LoRA (r, alpha, target_modules)
#   - PeftModel: wrapper que aplica los adaptadores LoRA a un modelo base congelado

from datetime import datetime
# Manejo de fechas y tiempos — para registrar timestamps (cuándo empezó el entrenamiento, etc.)

In [2]:
# Constants

BASE_MODEL = "meta-llama/Meta-Llama-3.1-8B"
FINETUNED_MODEL = f"ed-donner/pricer-2024-09-13_13.04.39"

# Hyperparameters para Fine-Tuning QLoRA

LORA_R = 32
LORA_ALPHA = 64
TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj"]

### Inicia sesión en HuggingFace

Si aún no tienes una cuenta de HuggingFace, visita https://huggingface.co para registrarte y crear un token.

Luego, selecciona los secretos para este cuaderno haciendo clic en el ícono de la llave a la izquierda y agrega un nuevo secreto llamado `HF_TOKEN` con el valor como tu token.

In [3]:
# Log in to HuggingFace

hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

## Probando diferentes Cuantizaciones


In [5]:
# Cargar el modelo base sin cuantizar

base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, device_map="auto")

config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

In [6]:
print(f"Impacto en la memoria: {base_model.get_memory_footprint() / 1e9:,.1f} GB")

Impacto en la memoria: 32.1 GB


In [5]:
base_model

NameError: name 'base_model' is not defined

## ¡Reinicia tu sesión!

Para cargar el siguiente modelo y limpiar la memoria caché del último modelo, ahora tendrás que ir a Runtime >> Reiniciar sesión y ejecutar las celdas iniciales (importaciones e inicio de sesión de HuggingFace) nuevamente.

Esto es para limpiar la GPU.

In [4]:
# Cargar el modelo base en 8 bits

quant_config = BitsAndBytesConfig(load_in_8bit=True)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [5]:
print(f"Impacto en Memoria: {base_model.get_memory_footprint() / 1e9:,.1f} GB")

Impacto en Memoria: 9.1 GB


In [9]:
base_model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear8bitLt(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear8bitLt(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear8bitLt(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear8bitLt(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear8bitLt(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear8bitLt(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear8bitLt(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (

## ¡Reinicia tu sesión!

Para cargar el siguiente modelo y limpiar la memoria caché del último modelo, ahora tendrás que ir a Runtime >> Reiniciar sesión y ejecutar las celdas iniciales (importaciones e inicio de sesión de HuggingFace) nuevamente.

Esto es para limpiar la GPU.

In [10]:
# !pip install --upgrade transformers bitsandbytes accelerate peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 81.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 784.9/784.9 kB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 78.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 80.1 MB/s eta 0:00:00
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.5.1
    Uninstalling hf-xet-1.5.1:
      Successfully uninstalled hf-xet-1.5.1
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.19.1
    Uninstalling tokenizers-0.19.1:
      Successfully uninstalled tokenizers-0.19.1
  Attempting uninstall: transformers
    Found existing installation: transformers 4.43.1
 

In [4]:
# Cargar el Tokenizer y el modelo Base en 4 bit

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4")

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    #device_map="auto",
)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [5]:
print(f"Impacto en memoria: {base_model.get_memory_footprint() / 1e9:,.2f} GB")

Impacto en memoria: 5.59 GB


In [6]:
base_model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): LlamaRM

In [7]:
fine_tuned_model = PeftModel.from_pretrained(base_model, FINETUNED_MODEL)

adapter_config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B /  109MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

In [8]:
print(f"Impacto en memoria: {fine_tuned_model.get_memory_footprint() / 1e9:,.2f} GB")

Impacto en memoria: 5.70 GB


In [9]:
fine_tuned_model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 4096)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

In [10]:
# Cada uno de los módulos de destino tiene 2 matrices de adaptador LoRA, llamadas lora_A y lora_B
# Están diseñadas para que los pesos se puedan adaptar sumando alpha * lora_A * lora_B
# Contemos la cantidad de pesos usando sus dimensiones:

# Vea las dimensiones de la matriz anterior
lora_q_proj = 4096 * 32 + 4096 * 32
lora_k_proj = 4096 * 32 + 1024 * 32
lora_v_proj = 4096 * 32 + 1024 * 32
lora_o_proj = 4096 * 32 + 4096 * 32

# Cada capa utiliza pues
lora_layer = lora_q_proj + lora_k_proj + lora_v_proj + lora_o_proj

# Hay en total 32 capas
params = lora_layer * 32

# Por tanto el total en Mb es:
size = (params * 4) / 1_000_000

print(f"Número total de parámetros: {params:,} y tamaño {size:,.1f}MB")

Número total de parámetros: 27,262,976 y tamaño 109.1MB


In [11]:
"""
Cálculo de parámetros entrenables en LoRA para Llama 3.1 8B
Semana 7 - Día 1: Fine-tuning con LoRA/QLoRA
"""

# ============================================================================
# 1. DIMENSIONES DE CADA MÓDULO OBJETIVO (q_proj, k_proj, v_proj, o_proj)
# ============================================================================

print("=" * 70)
print("CÁLCULO DE PARÁMETROS ENTRENABLES EN LoRA")
print("=" * 70)
print()

# Rango LoRA (r) = 32
r = 32

# Proyecciones de atención en Llama 3.1 8B
print("1. DIMENSIONES DE CADA MÓDULO DE ATENCIÓN:")
print("-" * 70)

# q_proj: 4096 → 4096 (proyección de Query)
q_in, q_out = 4096, 4096
lora_q_proj_a = q_in * r      # lora_A: (4096, 32)
lora_q_proj_b = r * q_out     # lora_B: (32, 4096)
lora_q_proj = lora_q_proj_a + lora_q_proj_b
print(f"q_proj:  {q_in} → {r} + {r} → {q_out} = {lora_q_proj_a:,} + {lora_q_proj_b:,} = {lora_q_proj:,}")

# k_proj: 4096 → 1024 (proyección de Key, con Grouped Query Attention)
k_in, k_out = 4096, 1024
lora_k_proj_a = k_in * r
lora_k_proj_b = r * k_out
lora_k_proj = lora_k_proj_a + lora_k_proj_b
print(f"k_proj:  {k_in} → {r} + {r} → {k_out} = {lora_k_proj_a:,} + {lora_k_proj_b:,} = {lora_k_proj:,}")

# v_proj: 4096 → 1024 (proyección de Value, con Grouped Query Attention)
v_in, v_out = 4096, 1024
lora_v_proj_a = v_in * r
lora_v_proj_b = r * v_out
lora_v_proj = lora_v_proj_a + lora_v_proj_b
print(f"v_proj:  {v_in} → {r} + {r} → {v_out} = {lora_v_proj_a:,} + {lora_v_proj_b:,} = {lora_v_proj:,}")

# o_proj: 4096 → 4096 (proyección de salida de atención)
o_in, o_out = 4096, 4096
lora_o_proj_a = o_in * r
lora_o_proj_b = r * o_out
lora_o_proj = lora_o_proj_a + lora_o_proj_b
print(f"o_proj:  {o_in} → {r} + {r} → {o_out} = {lora_o_proj_a:,} + {lora_o_proj_b:,} = {lora_o_proj:,}")

print()

# ============================================================================
# 2. PARÁMETROS POR CAPA DECODER
# ============================================================================

print("2. PARÁMETROS POR CAPA DECODER (estos 4 módulos × 1 capa):")
print("-" * 70)

lora_layer = lora_q_proj + lora_k_proj + lora_v_proj + lora_o_proj
print(f"Suma:    {lora_q_proj:,} + {lora_k_proj:,} + {lora_v_proj:,} + {lora_o_proj:,}")
print(f"Total por capa:  {lora_layer:,} parámetros")

print()

# ============================================================================
# 3. PARÁMETROS TOTALES EN LAS 32 CAPAS
# ============================================================================

print("3. PARÁMETROS TOTALES (32 capas):")
print("-" * 70)

num_layers = 32
total_params = lora_layer * num_layers
print(f"{lora_layer:,} × {num_layers} capas = {total_params:,} parámetros")

print()

# ============================================================================
# 4. TAMAÑO EN MEMORIA (en diferentes formatos)
# ============================================================================

print("4. TAMAÑO EN MEMORIA:")
print("-" * 70)

# FP32 (32 bits = 4 bytes por parámetro)
size_bytes = total_params * 4
size_mb = size_bytes / 1_000_000
size_gb = size_bytes / 1_000_000_000

print(f"FP32 (32 bits):  {total_params:,} × 4 bytes = {size_mb:.2f} MB = {size_gb:.3f} GB")

# FP16 (16 bits = 2 bytes por parámetro)
size_fp16_mb = (total_params * 2) / 1_000_000
print(f"FP16 (16 bits):  {total_params:,} × 2 bytes = {size_fp16_mb:.2f} MB")

# INT8 (8 bits = 1 byte por parámetro)
size_int8_mb = (total_params * 1) / 1_000_000
print(f"INT8 (8 bits):   {total_params:,} × 1 byte  = {size_int8_mb:.2f} MB")

print()

# ============================================================================
# 5. COMPARACIÓN: MODELO BASE vs LoRA
# ============================================================================

print("5. COMPARACIÓN: MODELO BASE vs LoRA:")
print("-" * 70)

base_model_params = 8_000_000_000  # 8 mil millones
base_model_mb = (base_model_params * 4) / 1_000_000
base_model_gb = base_model_mb / 1_000

percentage = (total_params / base_model_params) * 100

print(f"Modelo base (Llama 3.1 8B):")
print(f"  - Parámetros totales: {base_model_params:,} (8B)")
print(f"  - Tamaño FP32: {base_model_mb:,.0f} MB = {base_model_gb:.1f} GB")
print(f"  - % entrenable: 0% (congelado)")
print()
print(f"Adaptadores LoRA:")
print(f"  - Parámetros entrenables: {total_params:,} (~27.3M)")
print(f"  - Tamaño FP32: {size_mb:.2f} MB")
print(f"  - % del modelo base: {percentage:.3f}%")
print()
print(f"Combinado (Base + LoRA):")
print(f"  - Con modelo base en FP32: {base_model_gb:.1f} GB + {size_gb:.3f} GB ≈ {(base_model_gb + size_gb):.2f} GB")
print(f"  - Con modelo base en 4-bit (QLoRA): 4.0 GB + {size_gb:.3f} GB ≈ {(4.0 + size_gb):.2f} GB")

print()

# ============================================================================
# 6. VISUALIZACIÓN: PROPORCIÓN DE PARÁMETROS
# ============================================================================

print("6. PROPORCIÓN DE PARÁMETROS ENTRENABLES:")
print("-" * 70)

bar_length = 50
filled = int((total_params / base_model_params) * bar_length)
bar = "█" * filled + "░" * (bar_length - filled)
print(f"LoRA:       [{bar}] {percentage:.3f}%")
print(f"Base (frozen): [...{bar_length-filled} bloques...] {100-percentage:.2f}%")

print()

# ============================================================================
# 7. RESUMEN: EFICIENCIA DE LoRA
# ============================================================================

print("7. RESUMEN: ¿POR QUÉ LoRA ES EFICIENTE?")
print("-" * 70)
print()
print(f"Con LoRA, solo entrenamos {total_params:,} parámetros (~27M)")
print(f"en lugar de {base_model_params:,} parámetros (8B)")
print()
print(f"Esto reduce:")
print(f"  ✓ Memoria de GPU: de 32 GB → 4 GB (4-bit base) + 109 MB (LoRA)")
print(f"  ✓ Tiempo de entrenamiento: ~10-20 horas en Colab T4")
print(f"  ✓ Gradientes almacenados: solo para 27M parámetros")
print(f"  ✓ Optimizador Adam: requiere 2× los parámetros")
print()
print(f"Mientras mantiene la mayoría de la capacidad del modelo base,")
print(f"porque los 27M parámetros de LoRA se aplican donde más importa:")
print(f"en las proyecciones Q, K, V, O de la atención (toma de decisiones).")
print()
print("=" * 70)

CÁLCULO DE PARÁMETROS ENTRENABLES EN LoRA

1. DIMENSIONES DE CADA MÓDULO DE ATENCIÓN:
----------------------------------------------------------------------
q_proj:  4096 → 32 + 32 → 4096 = 131,072 + 131,072 = 262,144
k_proj:  4096 → 32 + 32 → 1024 = 131,072 + 32,768 = 163,840
v_proj:  4096 → 32 + 32 → 1024 = 131,072 + 32,768 = 163,840
o_proj:  4096 → 32 + 32 → 4096 = 131,072 + 131,072 = 262,144

2. PARÁMETROS POR CAPA DECODER (estos 4 módulos × 1 capa):
----------------------------------------------------------------------
Suma:    262,144 + 163,840 + 163,840 + 262,144
Total por capa:  851,968 parámetros

3. PARÁMETROS TOTALES (32 capas):
----------------------------------------------------------------------
851,968 × 32 capas = 27,262,976 parámetros

4. TAMAÑO EN MEMORIA:
----------------------------------------------------------------------
FP32 (32 bits):  27,262,976 × 4 bytes = 109.05 MB = 0.109 GB
FP16 (16 bits):  27,262,976 × 2 bytes = 54.53 MB
INT8 (8 bits):   27,262,976 × 1 by